In [0]:
# =============================================================
# logging_utils
# Shared utility functions for pipeline audit logging
# =============================================================
import uuid
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType, LongType

# Table reference
processing_log_table = f"{catalog}.{schema}.processing_log"
ingestion_log_table  = f"{catalog}.{schema}.ingestion_log"
doc_sources_table    = f"{catalog}.{schema}.doc_sources"

In [0]:
# =============================================================
# INGESTION LOG
# Tracks URL fetching: web → Volume (docs/raw)
# =============================================================

def initialize_ingestion_log_table():
    """
    Creates the ingestion_log Delta table if it doesn't exist.
    Acts as a safety net in case the table was not created via DDL.
    """
    if not spark.catalog.tableExists(ingestion_log_table):
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {ingestion_log_table} (
                log_id              BIGINT GENERATED ALWAYS AS IDENTITY,
                doc_id              BIGINT NOT NULL,
                url                 STRING,
                fetch_timestamp     TIMESTAMP,
                status              STRING,
                file_name           STRING,
                file_size_kb        DOUBLE,
                error_message       STRING,
                CONSTRAINT ingestion_log_pk PRIMARY KEY (log_id)
            )
            USING DELTA
            COMMENT 'Audit log tracking URL fetch status for the docs ingestion pipeline'
        """)
        print(f"✨ Created ingestion log table: {ingestion_log_table}")
    else:
        print(f"✅ Ingestion log table exists: {ingestion_log_table}")


def write_ingestion_log(doc_id, url, status, file_name=None, file_size_kb=None, error_message=None, fetch_method=None):
    """
    Writes a single log entry to the ingestion_log Delta table.
    Also updates last_fetched in doc_sources on SUCCESS.

    Args:
        doc_id          (int): The doc_id from doc_sources
        url             (str): The URL that was fetched
        status          (str): 'SUCCESS' or 'FAILED'
        file_name       (str): Name of the saved file, None on failure
        file_size_kb    (float): Size of the saved file in KB, None on failure
        error_message   (str): Error details if status is 'FAILED', else None
    """
    schema = StructType([
        StructField("doc_id", LongType(), False),
        StructField("url", StringType(), True),
        StructField("fetch_timestamp", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("file_name", StringType(), True),
        StructField("file_size_kb", DoubleType(), True),
        StructField("error_message", StringType(), True),
        StructField("fetch_method", StringType(), True)
    ])

    log_entry = Row(
        doc_id=doc_id,
        url=url,
        fetch_timestamp=datetime.now(),
        status=status,
        file_name=file_name,
        file_size_kb=file_size_kb,
        error_message=error_message,
        fetch_method=fetch_method
    )

    log_df = spark.createDataFrame([log_entry], schema=schema)
    log_df.write.format("delta").mode("append").saveAsTable(ingestion_log_table)

    # Update last_fetched in doc_sources on success
    if status == "SUCCESS":
        spark.sql(f"""
            UPDATE workspace.ai_project.doc_sources
            SET last_fetched = current_timestamp()
            WHERE doc_id = {doc_id}
        """)


def deactivate_doc_source(file_name, reason="unknown"):
    """
    Marks a doc_source as inactive based on the ingestion log file_name match.
    
    Args:
        file_name (str): The file name to match against ingestion_log
        reason    (str): Reason for deactivation e.g. 'too_small', 'invalid_type'
    """
    try:
        spark.sql(f"""
            UPDATE {doc_sources_table} il
            SET active = false
            WHERE doc_id = (
                SELECT doc_id FROM {ingestion_log_table}
                WHERE file_name = '{file_name}'
                ORDER BY fetch_timestamp DESC
                LIMIT 1
            )
        """)
        logger.info(f"ℹ️ Deactivated doc_source for: {file_name} (reason: {reason})")
    except Exception as e:
        logger.warning(f"⚠️ Could not deactivate doc_source for {file_name}: {e}")

In [0]:
# =============================================================
# PROCESSING LOG
# Tracks file processing: Volume → Delta (book/doc chunks)
# =============================================================

def initialize_processing_log_table():
    if not spark.catalog.tableExists(processing_log_table):
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {processing_log_table} (
                log_id               STRING NOT NULL,
                file_name            STRING,
                source_type          STRING,
                processed_timestamp  TIMESTAMP,
                target_table         STRING,
                chunk_count          INT,
                status               STRING,
                error_message        STRING,
                CONSTRAINT processing_log_pk PRIMARY KEY (log_id)
            )
            USING DELTA
            COMMENT 'Audit log tracking file processing status across all ingestion pipelines'
        """)
        print(f"✨ Created log table: {processing_log_table}")
    else:
        print(f"✅ Log table exists: {processing_log_table}")


def write_processing_log(file_name, source_type, target_table, status, chunk_count=0, error_message=None):
    """
    Writes a single log entry to the processing_log Delta table.

    Args:
        file_name       (str): Name of the file being processed
        source_type     (str): Source domain e.g. 'books' or 'docs'
        target_table    (str): Target Delta table the chunks were written to
        status          (str): 'SUCCESS' or 'FAILED'
        chunk_count     (int): Number of chunks written (0 on failure)
        error_message   (str): Error details if status is 'FAILED', else None
    """
    schema = StructType([
        StructField("log_id", StringType(), False),
        StructField("file_name", StringType(), True),
        StructField("source_type", StringType(), True),
        StructField("processed_timestamp", TimestampType(), True),
        StructField("target_table", StringType(), True),
        StructField("chunk_count", IntegerType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True)
    ])

    log_entry = Row(
        log_id=str(uuid.uuid4()),
        file_name=file_name,
        source_type=source_type,
        processed_timestamp=datetime.now(),
        target_table=target_table,
        chunk_count=chunk_count,
        status=status,
        error_message=error_message
    )

    log_df = spark.createDataFrame([log_entry], schema=schema)
    log_df.write.format("delta").mode("append").saveAsTable(processing_log_table)

In [0]:
# Auto-initialize on %run
initialize_ingestion_log_table()
initialize_processing_log_table()